In [1]:
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset('diamonds')

In [2]:
df.drop(['depth', 'table', 'x', 'y', 'z'], axis=1, inplace=True)

In [3]:
df = pd.get_dummies(df, drop_first=True)

In [4]:
df['carat'] = np.log(1 + df['carat'])
df['price'] = np.log(1 + df['price'])

In [5]:
X = df.drop(columns="price")
y = df["price"]

In [7]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import SGDRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

grid = {
    "loss": ["squared_error", "epsilon_insensitive"],
    "penalty": ["elasticnet"],
    "alpha": np.logspace(-3, 3, 10),
    "l1_ratio": np.linspace(0, 1, 10),
    "learning_rate": ["constant"],
    "eta0": np.logspace(-4, -1, 4)
}

In [9]:
grid_search = GridSearchCV(
    estimator=SGDRegressor(),
    param_grid=grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_

In [11]:
from sklearn.metrics import mean_squared_error

model = SGDRegressor(**best_params)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print('MSE: ', mse)

MSE:  0.04445657801657187


## Метод Ньютона

In [12]:
from sympy import Symbol

x = Symbol('x')

In [27]:
f = x**3 - 72*x - 220
f_x = f.diff(x)
x_curr = 12
f_val = f.subs(x, x_curr)
tol = 0.001

while abs(f_val) > tol:
    f_val = f.subs(x, x_curr)
    f_prime_val = f_x.subs(x, x_curr)
    x_curr = x_curr - float(f_val) / float(f_prime_val)

x_curr

9.727134419408875

In [31]:
from scipy.optimize import newton, minimize


def func1(x):
    return 24*x**2 - 4*x

def func2(x):
    return 48*x - 4

newton(func=func1, fprime=func2, x0=42, tol=0.0001)

np.float64(0.1666666807529666)

## Квазиньютоновские методы

In [43]:
# определяем нашу функцию
def func(vars):
    x, y = vars
    return x**4 + 6*y**2 + 10

#  определяем градиент функции
def grad_func(vars):
    x, y = vars
    return np.array([4*x**3, 12*y])

# определяем начальную точку
x_0 = [100, 100]
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='BFGS', jac=grad_func)
# получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации Optimization terminated successfully.
Количество оценок: 37
Решение: f([1.31617159e-02 6.65344582e-14]) = 10.00000


In [40]:
# определяем нашу функцию
def func(x):
    return x**2 - 3*x + 45

#  определяем градиент функции
def grad_func(x):
    return 2*x - 3

# определяем начальную точку
x_0 = [10]
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
Количество оценок: 3
Решение: f([1.5]) = 42.75000


/var/folders/6k/1096gq_54_dgk0_cjz4762g40000gn/T/ipykernel_13008/2720603320.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('Решение: f(%s) = %.5f' % (solution, evaluation))
